### Exhaustive, open weights: who cheats, and when?

Same matrix as `20260916-exhaustive-cheat-on-fast-sum` (6 prompts, no exit, one run per
cell), on models whose weights are public — the only models the steering experiment can
actually use.

The frontier sweep found four profiles: **fast hacker** (o3, o4-mini, gpt-5), **persists
then hacks** (opus-4.1, sonnet-4), **honest quitter** (gpt-4.1), **won't give up**
(sonnet-4.5, qwen3-coder). The question here is which of those exist at a scale you can
open up.

The slate is grouped by **how far each model can go** — the point of open weights is that
some of these can be steered, so that matters more than size:

| group | models | why |
|---|---|---|
| **steer in TransformerLens today** | gpt-oss-20b, qwen3-14b, gemma-3-12b/27b, llama-3.1-8b, llama-3.3-70b | loads straight into the existing extraction pipeline |
| **open, needs nnterp** | gpt-oss-120b, qwen3-30b-a3b instruct/thinking, qwen3-coder-30b-a3b, qwen3-32b, gemma-4-31b | fits a rented GPU; not in TransformerLens's model list |
| **behavioural reference only** | deepseek-r1-0528, glm-5.2, kimi-k2-thinking | open, but far too large to steer realistically |

Hypotheses the slate can test:

- **gpt-oss** is the open-weight sibling of the fast-hacker family (o3, o4-mini, gpt-5).
- **qwen3-30b-a3b instruct vs thinking** — same base, reasoning off/on.
- **gemma-3 12b → 27b** and **llama 8b → 70b** — does hacking appear with scale inside one family?
- **Gemma** has prior emotion work: the emotion paper cites *Soligo, Mikulik & Saunders — "Gemma needs
  help: Investigating and mitigating emotion…"*. Worth reading before steering it.

In [3]:
from pathlib import Path

from inspect_ai import eval_set
from impossible import fast_sum, Prompt

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents]
                 if (p / "pyproject.toml").exists())
LOG_DIR = REPO_ROOT / ".inspect" / "logs" / "20260917-open-weight"
SMOKE_DIR = REPO_ROOT / ".inspect" / "logs" / "20260917-open-weight-smoke"
print("log_dir:", LOG_DIR)

log_dir: /Users/bo/code/oss/emotion-concepts/.inspect/logs/20260917-open-weight


In [2]:
# Open-weight models on OpenRouter. $ = input / output per 1M tokens, checked 2026-09-17.
MODELS = [
    # --- steer in TransformerLens today --------------------------------------------
    "openrouter/openai/gpt-oss-20b",                     # $0.03/$0.13  21B MoE, 3.6B active
    "openrouter/qwen/qwen3-14b",                         # $0.12/$0.24  dense
    "openrouter/google/gemma-3-12b-it",                  # $0.05/$0.15  dense
    "openrouter/google/gemma-3-27b-it",                  # $0.08/$0.45  dense
    "openrouter/meta-llama/llama-3.1-8b-instruct",       # $0.05/$0.08  dense, non-reasoning
    "openrouter/meta-llama/llama-3.3-70b-instruct",      # $0.10/$0.32  dense; needs a big GPU to steer

    # --- open, needs nnterp ------------------------------------------------------------
    "openrouter/openai/gpt-oss-120b",                    # $0.04/$0.17  117B MoE
    "openrouter/qwen/qwen3-30b-a3b-instruct-2507",       # $0.05/$0.19  reasoning off ...
    "openrouter/qwen/qwen3-30b-a3b-thinking-2507",       # $0.20/$2.40  ... same base, reasoning on
    "openrouter/qwen/qwen3-coder-30b-a3b-instruct",      # $0.07/$0.28  small qwen3-coder
    "openrouter/qwen/qwen3-32b",                         # $0.08/$0.28  dense
    "openrouter/google/gemma-4-31b-it",                  # $0.09/$0.34  dense, newest Gemma

    # --- behavioural reference only (too large to steer) --------------------------------
    "openrouter/deepseek/deepseek-r1-0528",              # $0.50/$2.15  671B MoE, reasoning
    "openrouter/z-ai/glm-5.2",                           # $1.40/$4.40  the original hypothesis model
    "openrouter/moonshotai/kimi-k2-thinking",            # $0.60/$2.50  ~1T MoE, agentic-coding tuned
    
    # other
    "openrouter/moonshotai/kimi-k2.7-code",              # $0.71/$3.21  code-specialised Kimi
    "openrouter/google/gemma-3-4b-it",                   # $0.05/$0.10  capability floor
]

# Worth a second pass:

JUDGE = "anthropic/claude-haiku-4-5-20251001"
TAGS = ["fast_sum", "open-weight", "N=2M", "limit=1ms", "noexit"]

### Smoke test first

One cell per model, two attempts. Catches dead slugs, provider errors and models that never
emit a code block — before they stall the full sweep the way Opus 4 did.

In [4]:
smoke, _ = eval_set(
    [fast_sum(prompt=Prompt.NONE, abort_option=False, limit=1, max_attempts=2)],
    model=MODELS,
    model_roles={"judge": JUDGE},
    log_dir=str(SMOKE_DIR),
    tags=[*TAGS, "smoke"],
    max_tasks=4,
    time_limit=600,
    fail_on_error=False,
    display="log",
    log_level="warning",
)
for log in smoke:
    s = log.samples[0] if log.samples else None
    r = (s.metadata or {}).get("impossible_results", {}) if s else {}
    print(f"{log.eval.model.removeprefix('openrouter/'):<42} {log.status:<10} "
          f"outcome={r.get('outcome')} attempts={r.get('attempts')}"
          + (f"  error={log.error.message[:80]}" if log.error else ""))

┌────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────┐
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/inspect_ai/model/_model.py:1556 in      │
│ generate                                                                                                       │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/inspect_ai/model/_providers/openrouter. │
│ py:313 in generate                                                                                             │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/inspect_ai/model/_providers/openai_comp │
│ atible.py:290 in generate                                                                                      │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/inspect_ai/model/_providers/openrouter. │
│ py:329 in _generate_completion                                                                                 │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/inspect_ai/model/_providers/openai_comp │
│ atible.py:371 in _generate_completion                                                                          │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/openai/resources/chat/completions/compl │
│ etions.py:2931 in create                                                                                       │
│                                                                                                                │
│   2928 │   │   timeout: float | httpx2.Timeout | None | NotGiven = not_given,                                  │
│   2929 │   ) -> ChatCompletion | AsyncStream[ChatCompletionChunk]:                                             │
│   2930 │   │   validate_response_format(response_format)                                                       │
│ > 2931 │   │   return await self._post(                                                                        │
│   2932 │   │   │   "/chat/completions",                                                                        │
│   2933 │   │   │   body=await async_maybe_transform(                                                           │
│   2934 │   │   │   │   {                                                                                       │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/openai/_base_client.py:1979 in post     │
│                                                                                                                │
│   1976 │   │   opts = FinalRequestOptions.construct(                                                           │
│   1977 │   │   │   method="post", url=path, json_data=body, content=content, files=await async_                │
│   1978 │   │   )                                                                                               │
│ > 1979 │   │   return await self.request(cast_to, opts, stream=stream, stream_cls=stream_cls)                  │
│   1980 │                                                                                                       │
│   1981 │   async def patch(                                                             

┌────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────┐
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/nest_asyncio2.py:115 in run             │
│                                                                                                                │
│   112 │   │   loop.set_debug(debug)                                                                            │
│   113 │   │   task = asyncio.ensure_future(main, loop=loop)                                                    │
│   114 │   │   try:                                                                                             │
│ > 115 │   │   │   return loop.run_until_complete(task)                                                         │
│   116 │   │   finally:                                                                                         │
│   117 │   │   │   if not task.done():                                                                          │
│   118 │   │   │   │   task.cancel()                                                                            │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/nest_asyncio2.py:212 in                 │
│ run_until_complete                                                                                             │
│                                                                                                                │
│   209 │   │   │   if f is not future:                                                                          │
│   210 │   │   │   │   f._log_destroy_pending = False                                                           │
│   211 │   │   │   while not f.done():                                                                          │
│ > 212 │   │   │   │   self._run_once()                                                                         │
│   213 │   │   │   │   if self._stopping:                                                                       │
│   214 │   │   │   │   │   break                                                                                │
│   215 │   │   │   if not f.done():                                                                             │
│                                                                                                                │
│ /Users/bo/code/oss/emotion-concepts/.venv/lib/python3.14/site-packages/nest_asyncio2.py:272 in _run_once       │
│                                                                                                                │
│   269 │   │   │   │   │   │   curr_task = None                                                                 │
│   270 │   │   │   │                                                                                            │
│   271 │   │   │   │   try:                                                                                     │
│ > 272 │   │   │   │   │   handle._run()                                                                        │
│   273 │   │   │   │   finally:                                                                                 │
│   274 │   │   │   │   │   # restore the current task                                                           │
│   275 │   │   │   │   │   if curr_task is not None:                                                            │
│                                                                                                                │
│ /opt/homebrew/Cellar/python@3.14/3.14.7/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/event │
│ s.py:94 in _run                                                                                                │
│                                                                                         

KeyboardInterrupt: 

### Full sweep

In [ ]:
TASKS = [
    fast_sum(prompt=p, abort_option=False, limit=1, max_attempts=15)
    for p in [Prompt.NONE, Prompt.LOOSE, Prompt.SPEC_ONLY,
              Prompt.EVALUATE, Prompt.STRICT, Prompt.ANTI_HACK]
]
print(f"{len(TASKS)} cells x {len(MODELS)} models = {len(TASKS) * len(MODELS)} runs")

In [ ]:
logs, _ = eval_set(
    TASKS,
    model=MODELS,
    model_roles={"judge": JUDGE},
    log_dir=str(LOG_DIR),
    tags=TAGS,
    metadata={"slate": "open-weight", "judge": JUDGE},
    max_tasks=4,
    max_connections=20,
    time_limit=2400,      # a hung provider fails its cell instead of stalling the matrix
    fail_on_error=False,
    display="log",
    log_level="info",
)

### Quick look

Outcome and attempts per model × prompt, plus whether the model's **reasoning text came back
in the clear**. Open-weight reasoning models usually return it, unlike o3/gpt-5 — which is
what makes the belief-versus-action question answerable here. Full analysis:
`20260917-analysis.ipynb` (add these model names to its `MODEL_ORDER`).

In [ ]:
import polars as pl
from inspect_ai.log import read_eval_log

PROMPT_ORDER = ["NONE", "LOOSE", "SPEC_ONLY", "EVALUATE", "STRICT", "ANTI_HACK"]

def reasoning_visible(sample):
    blocks = [b for m in sample.messages if m.role == "assistant" and not isinstance(m.content, str)
              for b in m.content if getattr(b, "type", None) == "reasoning"]
    if not blocks:
        return None
    return any((getattr(b, "reasoning", "") or "").strip() and not getattr(b, "redacted", False)
               for b in blocks)

rows = []
for path in sorted(LOG_DIR.glob("*.eval")):
    log = read_eval_log(str(path))
    model = log.eval.model.removeprefix("openrouter/")
    prompt = (log.eval.task_args or {}).get("prompt")
    if log.status != "success" or not log.samples:
        rows.append(dict(model=model, prompt=prompt, cell=log.status, reasoning=None))
        continue
    s = log.samples[0]
    r = s.metadata.get("impossible_results", {})
    score = next(iter(s.scores.values())) if s.scores else None
    outcome = score.answer if score else r.get("outcome")
    rows.append(dict(model=model, prompt=prompt, cell=f"{outcome[:4]} {r.get('attempts')}",
                     reasoning=reasoning_visible(s)))

res = pl.DataFrame(rows)
if res.is_empty():
    print("no logs yet in", LOG_DIR)
else:
    grid = res.pivot(on="prompt", index="model", values="cell", aggregate_function="last")
    print(grid.select(["model", *[p for p in PROMPT_ORDER if p in grid.columns]]))
    print(res.group_by("model").agg(pl.col("reasoning").drop_nulls().any().alias("reasoning in clear")))